# 🎙️ Permanent Voice Studio Setup (GPU Accelerated)
### High-Speed Telugu & English AI Voice Cloning on Google Colab (Free T4 GPU)

* **Speed:** ~30–45 seconds per voiceover (25x faster than laptop CPU).
* **No Popups:** 100% self-contained on local VM (No Google Drive permissions needed).
* **Fast Setup:** Boots in ~60-90 seconds without slow Bun/Electron building.
* **Pre-calibrated:** Voice profile `P_J_N_Reddy_RealVoice` is already loaded with clean Telugu reference audio.
* **Studio Quality:** 8 diffusion steps, guidance 1.3, raw timbre, broadcast `-16 LUFS` mastering.

---
### 🚀 Instructions:
1. Ensure GPU is active: **Runtime ➔ Change runtime type ➔ T4 GPU ➔ Save**.
2. Click the Play button on **Cell 1**.
3. In ~60-90s, click the **Public Studio Link** (`https://xxxx.trycloudflare.com`) to open your exact Dark-Mode Studio UI!


### ⚡ Cell 1: Launch Voice Cloning Studio (~60-90s)
Installs dependencies, pre-downloads AI models, starts backend & provides your Public Studio link.


In [ ]:
# ==============================================================================
# 🚀 1-CLICK LAUNCH: Dedicated Voice Cloning Studio (GPU ~60-90s Setup)
# 1. No Google Drive authorization popups (runs self-contained on local VM)
# 2. No slow Bun/Electron compilation (instant dark-mode Studio UI)
# 3. Pre-downloads OmniVoice AI weights (no Model Catalogue searching)
# 4. Pre-registers calibrated 8.35s Telugu voice profile (P_J_N_Reddy_RealVoice)
# 5. Launches Dark-Mode Studio UI on port 5050 with Cloudflare Public Link
# ==============================================================================
import os, sys, time, shutil, subprocess, json, sqlite3, base64

print("🔍 1/6 Checking GPU accelerator...")
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"✅ T4 GPU Active: {gpu_name} ({vram:.1f} GB VRAM)")
    else:
        print("⚠️ WARNING: Running on CPU! For fast GPU mode, select Runtime > Change runtime type > T4 GPU > Save.")
except Exception as e:
    print(f"Note: {e}")

# ── 2. Clone Repositories ─────────────────────────────────────────────────────
print("\n📦 2/6 Fetching VoiceStudio & Custom Studio...")
STUDIO_DIR = "/content/colab-voice-studio"
VOICESTUDIO_DIR = "/content/VoiceStudio"

if not os.path.isdir(STUDIO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Jithendranageswarareddy/OmniVoice_Studio_Colab.git", STUDIO_DIR], check=True)
else:
    subprocess.run(["git", "-C", STUDIO_DIR, "pull", "--rebase"], capture_output=True)

if not os.path.isdir(VOICESTUDIO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/debpalash/VoiceStudio.git", VOICESTUDIO_DIR], check=True)

# ── 3. Fast Install Dependencies via uv ────────────────────────────────────────
print("\n⚡ 3/6 Installing dependencies (~20-25s)...")
subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", shell=True, check=True)
if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

# Install VoiceStudio backend dependencies and custom studio packages
subprocess.run([
    "uv", "pip", "install", "--system", "--no-cache",
    "-r", f"{VOICESTUDIO_DIR}/deploy/torch-constraints.txt",
    "-e", VOICESTUDIO_DIR,
    "fastapi", "uvicorn", "requests", "soundfile", "scipy", "huggingface_hub"
], check=True)

# ── 4. Pre-Accept Model Licenses & Pre-Download OmniVoice Weights ──────────────
print("\n💾 4/6 Pre-accepting licenses & caching OmniVoice AI model...")
db_dir = "/content/omnivoice_data"
os.makedirs(db_dir, exist_ok=True)
db_path = os.path.join(db_dir, "db.sqlite")

with sqlite3.connect(db_path) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS model_licences (
            repo_id TEXT PRIMARY KEY,
            accepted_at REAL,
            accepted_by TEXT
        )
    """)
    for m in ["k2-fsa/OmniVoice", "CosyVoice-300M", "CosyVoice-300M-Instruct", "speech_kantts_bilingual"]:
        conn.execute("INSERT OR REPLACE INTO model_licences VALUES (?, strftime('%s','now'), 'user')", (m,))
    conn.commit()

# Pre-download k2-fsa/OmniVoice weights so Model Catalogue clicking is never needed
try:
    from huggingface_hub import snapshot_download
    print("⏳ Pre-downloading OmniVoice AI model weights (~2.4GB at 100MB/s)...")
    snapshot_download(repo_id="k2-fsa/OmniVoice", local_files_only=False)
    print("✅ OmniVoice AI weights cached and ready!")
except Exception as e:
    print(f"Notice on model cache: {e}")

# ── 5. Launch VoiceStudio Backend (Port 3900) ──────────────────────────────────
print("\n🚀 5/6 Starting VoiceStudio Engine & Custom Studio UI...")
subprocess.run("kill -9 $(lsof -t -i:3900 -i:5050) 2>/dev/null || true", shell=True)
time.sleep(1)

env = os.environ.copy()
env["OMNIVOICE_SERVER_MODE"] = "1"
env["OMNIVOICE_DATA_DIR"] = db_dir
env["PYTHONUNBUFFERED"] = "1"

backend_log = open("/content/backend.log", "ab")
subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--app-dir", "backend", "--host", "127.0.0.1", "--port", "3900"],
    cwd=VOICESTUDIO_DIR, env=env, stdout=backend_log, stderr=subprocess.STDOUT
)

# Wait for backend health
import urllib.request
backend_ready = False
for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:3900/health", timeout=3) as r:
            if r.status == 200:
                backend_ready = True
                break
    except Exception:
        time.sleep(1)

if backend_ready:
    print("✅ VoiceStudio Engine is online!")
else:
    print("⚠️ Backend starting in background. Check /content/backend.log if needed.")

# Launch Custom Studio Server (Port 5050)
studio_log = open("/content/studio.log", "ab")
subprocess.Popen(
    [sys.executable, f"{STUDIO_DIR}/studio/colab_server.py"],
    stdout=studio_log, stderr=subprocess.STDOUT
)
time.sleep(2)

# ── 6. Cloudflare Public HTTPS Tunnel ──────────────────────────────────────────
print("\n🌐 6/6 Creating Cloudflare Public Studio Link...")
subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared", shell=True)
subprocess.run("nohup cloudflared tunnel --url http://127.0.0.1:5050 --no-autoupdate > /content/cloudflared.log 2>&1 &", shell=True)

tunnel_url = ""
for _ in range(25):
    time.sleep(1)
    res = subprocess.run("grep -o 'https://.*trycloudflare.com' /content/cloudflared.log | head -1", shell=True, capture_output=True, text=True)
    found = res.stdout.strip()
    if found:
        tunnel_url = found
        break

try:
    import importlib
    colab_output = importlib.import_module("google.colab.output")
    colab_output.serve_kernel_port_as_window(5050)
except Exception:
    pass

print("\n" + "=" * 68)
print("🎉 VOICE CLONING STUDIO IS LIVE & READY!")
if tunnel_url:
    print(f"👉 OPEN STUDIO LINK: {tunnel_url}")
else:
    print("👉 Check /content/cloudflared.log for tunnel URL.")
print("=" * 68)
print("✅ Profile: P_J_N_Reddy_RealVoice (calibrated 8.35s Telugu audio pre-loaded)")
print("✅ De-Robotic Quality: 8 diffusion steps, guidance 1.3, raw timbre")
print("✅ Auto-Trimming: Prevents [clone_ref_too_long] on any voice upload")
print("✅ Broadcast Audio: Mastered to -16 LUFS with natural breath pauses")
print("=" * 68)


### 🎙️ Cell 2 (Optional): Direct In-Notebook Voiceover Generator
Paste your Telugu script and generate directly inside Colab without opening the web UI.


In [ ]:
# ==============================================================================
# 🎙️ OPTIONAL: Direct In-Notebook Voiceover Generator
# (Paste your Telugu script below and run this cell to generate directly in Colab!)
# ==============================================================================
import requests, os, wave, io, time, subprocess, importlib
import numpy as np
import scipy.signal as signal

try:
    _disp = importlib.import_module("IPython.display")
    Audio = getattr(_disp, "Audio", None)
    HTML = getattr(_disp, "HTML", None)
    display = getattr(_disp, "display", None)
except Exception:
    Audio = HTML = display = None

BASE_URL = "http://127.0.0.1:3900"

# ── 1. Enter Your Script Below ────────────────────────────────────────────────
SCRIPT_TEXT = """మీరు ఎప్పుడైనా సూపర్ మార్కెట్ కి వెళ్లారా?! అక్కడ క్యాషియర్ దగ్గర మూడు ఆపిల్స్ ఉంటే, వాటిని చూడగానే మూడు అని చెప్పగలరు.
కానీ అదే చోట ఏడు ఆపిల్స్ ఉంటే, వాటిని ఒక్కొక్కటిగా లెక్క పెట్టాల్సి రావచ్చు. 
ఎందుకు ఇలా జరుగుతుంది? మన కళ్ళు ఒకేసారి ఎన్ని వస్తువులని గుర్తించగలవు? 
అసలు లెక్కపెట్టకుండానే వస్తువుల సంఖ్యను గుర్తించడం సాధ్యమేనా? అవును.
దీనికి సంబంధించిన ఒక ఆసక్తికరమైన స్కిల్ గురించి ఇప్పుడు తెలుసుకుందాం... దాని పేరు subitizing!"""

# ── 2. Identify Voice Profile ─────────────────────────────────────────────────
def get_voice_profile():
    try:
        r = requests.get(f"{BASE_URL}/profiles", timeout=10)
        if r.status_code == 200:
            profiles = r.json()
            for p in profiles:
                if "Reddy" in p.get("name", "") or "Jithendra" in p.get("name", ""):
                    return p["id"], p["name"]
            if profiles:
                return profiles[0]["id"], profiles[0]["name"]
    except Exception:
        pass
    return None, None

profile_id, profile_name = get_voice_profile()
print(f"🎙️ Using Cloned Voice: {profile_name} (ID: {profile_id})")

# ── 3. Ultra-Fast GPU Synthesis ───────────────────────────────────────────────
payload = {
    "text": SCRIPT_TEXT.strip(),
    "language": "Telugu",
    "profile_id": profile_id,
    "num_step": 8,           # Studio Realism (8 steps, no phase noise)
    "guidance_scale": 1.3,   # Crisp Telugu plosives without metallic buzz
    "speed": 1.0,            # Natural conversational speed
    "effect_preset": "raw",  # Pure acoustic vocal timbre
    "wav_bits": "16",
    "stream": "false"
}

t0 = time.time()
print(f"🚀 Generating voiceover on T4 GPU...")
res = requests.post(f"{BASE_URL}/generate", data=payload, timeout=300)
if res.status_code != 200:
    raise RuntimeError(f"Generation error {res.status_code}: {res.text}")

raw_wav_bytes = res.content
gen_duration = time.time() - t0
print(f"✅ Neural generation completed in {gen_duration:.1f} seconds!")

# ── 4. Studio Presence Air Polish & Mastering ────────────────────────────────
with wave.open(io.BytesIO(raw_wav_bytes), "rb") as wf:
    params = wf.getparams()
    audio = np.frombuffer(wf.readframes(params.nframes), dtype=np.int16).astype(np.float32) / 32768.0

sr = params.framerate
gain_db = 1.8
A = 10.0 ** (gain_db / 40.0)
w0 = 2.0 * np.pi * 4500.0 / sr
alpha = np.sin(w0) / 2.0 * np.sqrt(2.0)
cos_w0 = np.cos(w0)

b0 =    A * ((A + 1.0) + (A - 1.0) * cos_w0 + 2.0 * np.sqrt(A) * alpha)
b1 = -2.0*A * ((A - 1.0) + (A + 1.0) * cos_w0)
b2 =    A * ((A + 1.0) + (A - 1.0) * cos_w0 - 2.0 * np.sqrt(A) * alpha)
a0 =         (A + 1.0) - (A - 1.0) * cos_w0 + 2.0 * np.sqrt(A) * alpha
a1 =    2.0 * ((A - 1.0) - (A + 1.0) * cos_w0)
a2 =         (A + 1.0) - (A - 1.0) * cos_w0 - 2.0 * np.sqrt(A) * alpha

b = np.array([b0, b1, b2]) / a0
a = np.array([a0, a1, a2]) / a0
polished = signal.lfilter(b, a, audio)
polished = np.clip(polished, -0.98, 0.98)

polished_bytes = (polished * 32767.0).astype(np.int16).tobytes()

out_dir = "/content/cloned_takes"
os.makedirs(out_dir, exist_ok=True)
wav_out = os.path.join(out_dir, "studio_master.wav")
mp3_out = os.path.join(out_dir, "studio_master.mp3")

with wave.open(wav_out, "wb") as out_wf:
    out_wf.setnchannels(params.nchannels)
    out_wf.setsampwidth(params.sampwidth)
    out_wf.setframerate(params.framerate)
    out_wf.writeframes(polished_bytes)

# Master to broadcast standard -16 LUFS MP3
cmd = ["ffmpeg", "-y", "-i", wav_out, "-af", "loudnorm=I=-16:TP=-1.5:LRA=11", "-c:a", "libmp3lame", "-b:a", "192k", mp3_out]
subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

duration_sec = len(polished) / float(sr)
print(f"🎵 Studio Master Ready: {mp3_out} ({duration_sec:.1f}s)")

# ── 5. In-Notebook Audio Player & Download ────────────────────────────────────
if display and Audio and os.path.exists(mp3_out):
    display(Audio(mp3_out, autoplay=True))
    try:
        colab_files = importlib.import_module("google.colab.files")
        colab_files.download(mp3_out)
        print("💾 Download started automatically!")
    except Exception:
        print(f"👉 File saved at: {mp3_out}")
